In [1]:
import pandas as pd

In [2]:
# df = pd.read_csv("path", encoding="cp949")
# 물품종별칼럼 = 'item', 부대칼럼 ='mil_unit' 있다고 가정 /
# 시작일 'yyyymmdd' 형식으로 입력 / period = 시작일부터 유찰율 보고싶은 일수

def bid_result_patio(item=None, mil_unit=None, start_date=None, period_days=None):
    global df
    filtered_df = df.copy()
    filtered_df['최종낙찰일자'] = pd.to_datetime(filtered_df['최종낙찰일자'], errors='coerce')

    if item is not None:
        if isinstance(item, str):
            filtered_df = filtered_df[filtered_df['item'] == item]
        else:
            filtered_df = filtered_df[filtered_df['item'].isin(item)]

    if mil_unit is not None:
        if isinstance(mil_unit, str):
            filtered_df = filtered_df[filtered_df['mil_unit'] == mil_unit]
        else:
            filtered_df = filtered_df[filtered_df['mil_unit'].isin(mil_unit)]

    if start_date is not None:
        start_date = pd.to_datetime(start_date, format='%Y%m%d')

        if period_days is not None:
            end_date = start_date + pd.Timedelta(days=period_days)

            # 최종낙찰일자 컬럼이 datetime 형식인지 확인 후 필터링
            filtered_df = filtered_df[
                (filtered_df['최종낙찰일자'] >= start_date) &
                (filtered_df['최종낙찰일자'] <= end_date)
            ]
        else:
            filtered_df = filtered_df[filtered_df['최종낙찰일자'] >= start_date]

    # 낙찰결과구분명별 개수 및 비율 계산
    result_counts = filtered_df['개찰결과구분명'].value_counts()
    total = result_counts.sum()
    result_ratio = (result_counts / total * 100).round(2)

    result_df = result_ratio.reset_index()
    result_df.columns = ['개찰결과구분명', '비율(%)']

    return result_df

In [3]:
# 예시데이터프레임
data = {
    'item': ['컴퓨터', '컴퓨터', '프린터', '컴퓨터', '프린터', '모니터', '모니터', '컴퓨터'],
    'mil_unit': ['1사단', '2사단', '1사단', '1사단', '2사단', '3사단', '1사단', '3사단'],
    '개찰결과구분명': ['유찰', '개찰완료', '순위확정', '유찰', '개찰완료', '유찰', '순위확정', '개찰완료'],
    '최종낙찰일자': ['2025-01-01', '2025-01-05', '2025-01-10', '2025-02-01', '2025-02-15', '2025-03-01', '2025-03-05', '2025-04-01']
}

df = pd.DataFrame(data)

bid_result_patio(mil_unit='1사단', start_date='20250101', period_days=365) 

,개찰결과구분명,비율(%)
0,유찰,50.0
1,순위확정,50.0
